# MetaCyc Pathway Name Decoder

## Project overview

This notebook retrieves readable names for a predefined list of MetaCyc pathway IDs using the BioCyc web service.

**Input:** `my_file.csv`(but use yours), containing a column named `pathway`  
**Output:** `MetaCyc_Pathway_Names.csv`, containing the pathway ID and its retrieved name.

The notebook is organized into two parts:

- **Development and inspection:** loading the IDs, authenticating, testing the API, and inspecting the XML response.
- **Implementation:** defining the decoder function, retrieving all names, and exporting the results.

> Keep the input CSV in the same working directory as this notebook, or update `input_csv` below. BioCyc credentials are entered interactively and are not stored in this notebook.


## 1. Import libraries

In [ ]:
import csv
import getpass
import time
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
import requests

## 2. Load pathway IDs

In [ ]:
# Input file and required column
input_csv = Path("clr_50_heatmap.csv")
pathway_column = "pathway"

if not input_csv.is_file():
    raise FileNotFoundError(f"Input file not found: {input_csv.resolve()}")

# Read pathway IDs from the CSV
pl = []

with input_csv.open(mode="r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    if pathway_column not in (reader.fieldnames or []):
        raise ValueError(
            f"Required column {pathway_column!r} was not found in {input_csv.name}"
        )

    for row in reader:
        pathway_id = (row.get(pathway_column) or "").strip()
        if pathway_id:
            pl.append(pathway_id)

if not pl:
    raise ValueError("No pathway IDs were found in the input CSV.")

print(f"Pathway IDs loaded: {len(pl)}")
print(pl)


## 3. Authenticate with BioCyc

Enter the email address and password associated with your BioCyc account when prompted. The password is hidden while typing.

Do not replace this interactive prompt with a hard-coded password, and do not save credentials or session cookies in the notebook.


In [ ]:
session = requests.Session()

email = input("BioCyc registered email: ").strip()
password = getpass.getpass("BioCyc password: ")

login_url = "https://websvc.biocyc.org/credentials/login/"

login_response = session.post(
    login_url,
    data={"email": email, "password": password},
    timeout=30
)

login_response.raise_for_status()

print("Login request completed.")
print("HTTP status:", login_response.status_code)
print("Session cookies received:", bool(session.cookies))


## 4. Test the API connection

An HTTP status of 200 does not by itself guarantee that the response is XML. For example, an authentication page can also be returned with status 200. This test therefore checks the response content type before attempting to parse it.


In [ ]:
test_response = session.get(
    "https://websvc.biocyc.org/getxml",
    params={
        "id": "META:PWY-3781",
        "detail": "full"
    },
    timeout=30
)

test_response.raise_for_status()

content_type = test_response.headers.get("Content-Type", "")
print("HTTP status:", test_response.status_code)
print("Content type:", content_type)
print("Response URL:", test_response.url)

if "xml" not in content_type.lower():
    raise ValueError(
        f"Expected an XML response from BioCyc, received {content_type!r}. "
        "Check the authentication session and API response."
    )

print("\nResponse preview:")
print(test_response.text[:500])


## 5. Inspect the XML response

In [ ]:
# Parse the test response and locate the Pathway element
test_root = ET.fromstring(test_response.content)
test_pathway = test_root.find(".//Pathway")

if test_pathway is None:
    raise ValueError("The XML response did not contain a Pathway element.")

print("Root tag:", test_root.tag)
print("\nPathway attributes:")
print(test_pathway.attrib)

print("\nDirect child elements:")
for child in test_pathway:
    print(child.tag, child.attrib)


### 5.1 Explore pathway annotations

This exploratory cell displays selected XML elements. It is included to document the XML structure; it is not required for the batch retrieval itself.


In [ ]:
for tag in ["common-name", "comment", "taxonomic-range"]:
    print(f"\n--- {tag.upper()} ---")

    for element in test_pathway.findall(f".//{tag}"):
        element_text = " ".join("".join(element.itertext()).split())
        if element_text:
            print(element_text)


### 5.2 Extract the direct pathway name

Use `find("common-name")` to select the pathway's direct `common-name` child. A broad descendant search such as `find(".//common-name")` can return a nested common name belonging to another annotation instead.


In [ ]:
test_common_name = test_pathway.find("common-name")

if test_common_name is not None:
    test_name = "".join(test_common_name.itertext()).strip()
    print("Pathway name:", test_name or "Empty pathway name")
else:
    print("No direct pathway name found.")


## 6. Define the pathway-name decoder

The function below:

1. Requests the XML record for one MetaCyc pathway ID.
2. Checks the HTTP response and verifies that the response is XML.
3. Parses the XML and locates the `Pathway` element.
4. Extracts the direct `common-name` element.
5. Returns the pathway name, or `None` if no name is available.

The function expects IDs without the `META:` prefix, as in the input CSV.


In [ ]:
def get_pathway_name(pathway_id):
    """Retrieve the readable name for one MetaCyc pathway ID."""

    response = session.get(
        "https://websvc.biocyc.org/getxml",
        params={
            "id": f"META:{pathway_id}",
            "detail": "full"
        },
        timeout=30
    )

    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "")
    if "xml" not in content_type.lower():
        raise ValueError(
            f"Expected XML for {pathway_id}, received {content_type!r}"
        )

    try:
        root = ET.fromstring(response.content)
    except ET.ParseError as error:
        raise ValueError(
            f"Could not parse the XML response for {pathway_id}"
        ) from error

    pathway = root.find(".//Pathway")
    if pathway is None:
        return None

    common_name = pathway.find("common-name")
    if common_name is None:
        return None

    pathway_name = "".join(common_name.itertext()).strip()
    return pathway_name or None


## 7. Retrieve names for all pathway IDs

In [ ]:
pathway_names = []

for pathway_id in pl:
    try:
        name = get_pathway_name(pathway_id)

        pathway_names.append({
            "Pathway_ID": pathway_id,
            "Pathway_Name": name,
            "Status": "success" if name else "name_not_found",
            "Error": None
        })

        print(f"{pathway_id}: {name if name else 'No pathway name found'}")

    except Exception as error:
        pathway_names.append({
            "Pathway_ID": pathway_id,
            "Pathway_Name": None,
            "Status": "failed",
            "Error": str(error)
        })

        print(f"{pathway_id}: Failed - {error}")

    # Pause between requests
    time.sleep(1)

df = pd.DataFrame(pathway_names)
df.head()


## 8. Export the results

The output includes a status and error column so that a missing pathway name can be distinguished from a failed API request.


In [ ]:
output_csv = Path("Sample_Metacyc_Pathway_Names.csv")

df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Results saved to: {output_csv.resolve()}")
print("\nRetrieval summary:")
print(df["Status"].value_counts(dropna=False))


In [ ]:
#-------------------------TEST-------------------------#

for pathway_id in test_ids:
    try:
        name = get_pathway_name(pathway_id)
        print(f"{pathway_id}: {name}")

    except Exception as error:
        print(f"{pathway_id}: Failed - {error}")